In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings(action='ignore')

In [11]:
import os
# # 노트북 파일이 있는 폴더로 이동 (예시)
os.chdir(r'C:\Users\Admin\hipython\ml\data')

# # 변경 후 확인
print("변경 후:", os.getcwd())

변경 후: C:\Users\Admin\hipython\ml\data


In [40]:
X = pd.read_csv('C:\\Users\\Admin\\hipython\\ml\\data\\X.csv', encoding='euc-kr')
y = pd.read_csv('C:\\Users\\Admin\\hipython\\ml\\data\\y.csv', encoding='euc-kr')


In [25]:
X.head()

,cust_id,총구매액,최대구매액,환불금액,주구매상품,주구매지점,내점일수,내점당구매건수,주말방문비율,구매주기
0,0,68282840,11264000,6860000.0,기타,강남점,19,3.894737,0.527027,17
1,1,2136000,2136000,300000.0,스포츠,잠실점,2,1.500000,0.000000,1
2,2,3197000,1639000,NaN,남성 캐주얼,관악점,2,2.000000,0.000000,1
3,3,16077620,4935000,NaN,기타,광주점,18,2.444444,0.318182,16
4,4,29050000,24000000,NaN,보석,본 점,2,1.500000,0.000000,85


In [26]:
y.head()

,cust_id,gender
0,0,0
1,1,0
2,2,1
3,3,1
4,4,0


In [28]:
X.info()
X.describe()

<class 'pandas.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cust_id  3500 non-null   int64  
 1   총구매액     3500 non-null   int64  
 2   최대구매액    3500 non-null   int64  
 3   환불금액     1205 non-null   float64
 4   주구매상품    3500 non-null   str    
 5   주구매지점    3500 non-null   str    
 6   내점일수     3500 non-null   int64  
 7   내점당구매건수  3500 non-null   float64
 8   주말방문비율   3500 non-null   float64
 9   구매주기     3500 non-null   int64  
dtypes: float64(3), int64(5), str(2)
memory usage: 273.6 KB


,cust_id,총구매액,최대구매액,환불금액,내점일수,내점당구매건수,주말방문비율,구매주기
count,3500.000000,3.500000e+03,3.500000e+03,1.205000e+03,3500.000000,3500.000000,3500.000000,3500.000000
mean,1749.500000,9.191925e+07,1.966424e+07,2.407822e+07,19.253714,2.834963,0.307246,20.958286
std,1010.507298,1.635065e+08,3.199235e+07,4.746453e+07,27.174942,1.912368,0.289752,24.748682
min,0.000000,-5.242152e+07,-2.992000e+06,5.600000e+03,1.000000,1.000000,0.000000,0.000000
25%,874.750000,4.747050e+06,2.875000e+06,2.259000e+06,2.000000,1.666667,0.027291,4.000000
50%,1749.500000,2.822270e+07,9.837000e+06,7.392000e+06,8.000000,2.333333,0.256410,13.000000
75%,2624.250000,1.065079e+08,2.296250e+07,2.412000e+07,25.000000,3.375000,0.448980,28.000000
max,3499.000000,2.323180e+09,7.066290e+08,5.637530e+08,285.000000,22.083333,1.000000,166.000000


In [29]:
y.info()
y.describe()

<class 'pandas.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   cust_id  3500 non-null   int64
 1   gender   3500 non-null   int64
dtypes: int64(2)
memory usage: 54.8 KB


,cust_id,gender
count,3500.000000,3500.000000
mean,1749.500000,0.376000
std,1010.507298,0.484449
min,0.000000,0.000000
25%,874.750000,0.000000
50%,1749.500000,0.000000
75%,2624.250000,1.000000
max,3499.000000,1.000000


In [30]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [46]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, f1_score

In [50]:
# ---------------------------------------------------------
# 2. [정확도 향상 팁 1] 데이터 병합 (Merge)
# ---------------------------------------------------------
# X와 y를 cust_id 기준으로 합쳐서 데이터 정합성을 맞춥니다.
# 그 후 불필요한 cust_id는 바로 제거합니다.
df = pd.merge(X, y, on='cust_id').drop(columns=['cust_id'])

# 3. 결측치 처리
df['환불금액'] = df['환불금액'].fillna(0)

# ---------------------------------------------------------
# 4. [정확도 향상 팁 2] 수치형 데이터 로그 변환 (Log Transform)
# ---------------------------------------------------------
# '총구매액'처럼 숫자의 단위가 매우 크고 왜곡된(Skewed) 데이터는 
# 로그 변환을 통해 정규분포에 가깝게 만들어주면 모델 성능이 올라가는 경우가 많습니다.
monetary_cols = ['총구매액', '최대구매액']
for col in monetary_cols:
    # 음수가 있을 수 있으므로(환불 등) 최솟값을 더해 0 이상의 값으로 만든 후 로그를 취합니다.
    df[col] = np.log1p(df[col] - df[col].min())

# 5. 범주형 데이터 인코딩
le = LabelEncoder()
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# 6. 피처(X)와 타겟(y) 분리
X_final = df.drop(columns=['gender'])
y_final = df['gender']

# 7. 표준화 (StandardScaler)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)
X_final = pd.DataFrame(X_scaled, columns=X_final.columns)

# ---------------------------------------------------------
# 8. 모델 학습 및 비교
# ---------------------------------------------------------
models = {
    "LogisticRegression": LogisticRegression(),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBoost(GBC)": GradientBoostingClassifier(random_state=42)
}

results_f1 = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results_f1[name] = f1
    print(f"{name:25} | {acc:.4f}     | {f1:.4f}")

# 5. F1 Score 기준 짱짱맨 선정
best_f1_model = max(results_f1, key=results_f1.get)


print("-" * 55)
print(f"🏆 F1 Score 기준 짱짱맨: [{best_f1_model}]")
print(f"최고 F1 점수: {results_f1[best_f1_model]:.4f}")

LogisticRegression        | 0.6314     | 0.2670
DecisionTree              | 0.5643     | 0.4256
RandomForest              | 0.6300     | 0.4046
XGBoost(GBC)              | 0.6386     | 0.4075
-------------------------------------------------------
🏆 F1 Score 기준 짱짱맨: [DecisionTree]
최고 F1 점수: 0.4256
